In [ ]:
import pandas as pd
import numpy as np
import re 
from pathlib import Path


In [ ]:
df = pd.read_csv("../data/BMW/eng_ger_auto_reviews_dataset.csv", encoding="utf-8")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
#Dataset overview
df.info()

### Remove Duplicate Reviews

Duplicate reviews were identified during the EDA phase.
We remove them based on the unique reviewId.

In [ ]:
df = df.drop_duplicates(subset="reviewId").reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

### Remove Missing Reviews

Reviews without textual content cannot be used for NLP analysis.

In [ ]:
df = df.dropna(subset=["content"]).reset_index(drop=True)

In [ ]:
#Check
df.isna().sum()

In [ ]:
#Spalten prüfen - Verstehen welche Metadaten vorhanden sind
df.columns

### Remove Unnecessary Metadata Columns

Some metadata columns are not relevant for text analysis.

In [ ]:
cols_to_drop = ["userName", "userImage", "replyContent", "repliedAt"]

df = df.drop(columns=cols_to_drop)

In [ ]:
#Check
df.head()

##### Wir entfernen im nächsten Schritt html

In [ ]:
import re 
def remove_html(text):
    return re.sub(r"<.*?>", "", text)
df["content"]=df["content"].apply(remove_html)
df.head()

### Normalize Text (Lowercase)

In [ ]:
df["content"] = df["content"].str.lower()

### Normalize Whitespace

In [ ]:
df["content"] = df["content"].str.replace(r"\s+", " ", regex=True).str.strip()

### Remove Very Short Reviews

In [ ]:
df = df[df["content"].str.split().str.len() >= 2]

### Convert Timestamp

In [ ]:
df["at"]=pd.to_datetime(df["at"], errors="coerce")

In [ ]:
df.info()

In [ ]:
df.head()

### Final Quality Check

In [ ]:
print("Final dataset shape:", df.shape)

df.isna().sum()

### Save Clean Dataset

In [ ]:
df.to_csv("../data/BMW/clean_reviews.csv", index=False)

##### Hinweis: Nach Projektinternem Austausch am 10.03.2026
Nach Diskussion mit Martin Lohr, haben wir uns dazu entschieden, keine Tokenization manuell vorzunehmen, da das SentenceTransformer-Modell (all-MiniLM-L6-v2) die tokenization intern übernimmt. Wenn wir manuell tokenisieren, würden wir die semantische Struktur zerstören und die Embeddings schlechter machen. Somit ist eine Textbereinigung mit HTML entfernen, lowercasing, Interpunktion löschen ausreichend.

## Preprocessing Summary

In diesem Notebook wurde eine strukturierte Datenbereinigung durchgeführt, um die Rohdaten für die nachfolgende Analyse mit Sentence Transformer Modellen vorzubereiten.

Der Preprocessing-Prozess begann mit dem Laden des ursprünglichen Datensatzes mit App-Reviews. Anschließend wurden mehrere Cleaning-Schritte durchgeführt, um die Datenqualität zu verbessern und potenzielle Störungen für spätere Analysen zu reduzieren.

### 1. Entfernung von Duplikaten
Zunächst wurden doppelte Reviews anhand der eindeutigen `reviewId` entfernt. Duplikate können die Analyse verzerren, da identische Inhalte mehrfach in statistische Auswertungen oder Embedding-Berechnungen eingehen würden.

### 2. Entfernung fehlender Inhalte
Reviews ohne Textinhalt (`content`) wurden gelöscht, da diese keinen semantischen Informationsgehalt besitzen und somit für NLP-Modelle nicht verwendbar sind.

### 3. Entfernen nicht benötigter Metadaten
Einige Spalten wie `userName`, `userImage`, `replyContent` und `repliedAt` wurden entfernt. Diese Informationen sind für die geplante Textanalyse und Embedding-Generierung nicht relevant und würden lediglich den Datensatz unnötig vergrößern.

### 4. Entfernen von HTML-Artefakten
HTML-Tags wurden aus den Review-Texten entfernt, da diese beim Web-Scraping entstehen können und keine semantische Bedeutung für die Textanalyse besitzen.

### 5. Normalisierung von Whitespace
Mehrfache Leerzeichen, Zeilenumbrüche und ähnliche Formatierungsartefakte wurden vereinheitlicht. Dies stellt sicher, dass die Texte in konsistenter Form vorliegen.

### 6. Entfernung sehr kurzer Reviews
Extrem kurze Reviews wurden entfernt, da sie häufig wenig semantische Information enthalten (z. B. „good“, „ok“ oder „nice“) und daher nur begrenzten analytischen Mehrwert bieten.

Nach diesen Schritten wurde der bereinigte Datensatz als `clean_reviews.csv` gespeichert.

---

## Gründe für bewusst ausgelassene Preprocessing-Schritte

Bestimmte klassische NLP-Preprocessing-Methoden wurden bewusst **nicht angewendet**, da moderne Transformer-Modelle mit natürlicher Sprache besser umgehen können, wenn der ursprüngliche Kontext erhalten bleibt.

### Stopword-Entfernung wurde ausgelassen
Stopwords wie „the“, „and“ oder „is“ können für Transformer-Modelle wichtige Kontextinformationen enthalten. Das Entfernen dieser Wörter kann den semantischen Zusammenhang eines Satzes verändern.

### Lemmatization und Stemming wurden nicht durchgeführt
Transformer-Modelle nutzen Subword-Tokenisierung und sind darauf trainiert, verschiedene Wortformen zu verstehen. Eine vorherige Normalisierung der Wortformen ist daher meist nicht notwendig und kann sogar Informationen verlieren.

### Tokenization wurde nicht manuell durchgeführt
Modelle wie Sentence Transformers besitzen eine integrierte Tokenisierung, die exakt auf das jeweilige Modell abgestimmt ist. Eine externe Tokenisierung wäre daher redundant.

### Punctuation wurde nicht entfernt
Satzzeichen können zusätzliche semantische Hinweise liefern, beispielsweise Emotionen oder Betonung (z. B. „!“ oder „?“). Da Transformer-Modelle mit solchen Informationen umgehen können, wurden diese bewusst beibehalten.

---

## Ergebnis

Das Ergebnis des Preprocessing-Schritts ist ein bereinigter Datensatz, der strukturelle Probleme entfernt, gleichzeitig aber den natürlichen Sprachkontext der Reviews bewahrt. Dadurch wird sichergestellt, dass die nachfolgenden Analysen und Embedding-Berechnungen mit Sentence Transformer Modellen auf qualitativ hochwertigen und semantisch vollständigen Textdaten basieren.

In [1]:
# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np

In [2]:
# =========================
# 2. Load Data
# =========================
df = pd.read_csv("../data/BMW/eng_ger_auto_reviews_dataset.csv")

print("Original shape:", df.shape)

Original shape: (21721, 12)


In [3]:
# =========================
# 3. Keep only relevant columns
# =========================
df = df[["content", "score", "at"]]

In [4]:
# =========================
# 4. Rename columns (STANDARDIZATION)
# =========================
df = df.rename(columns={
    "content": "text",
    "score": "rating",
    "at": "date"
})

In [5]:
# =========================
# 5. Drop missing values
# =========================
df = df.dropna(subset=["text", "rating"])

In [6]:
# =========================
# 6. Clean text
# =========================
df["text"] = df["text"].astype(str)
df["text"] = df["text"].str.strip()

In [7]:
# Remove very short reviews
df = df[df["text"].str.split().str.len() >= 2]
#Hier kann man optimierung vornehmen um noise zu reduzieren im LLM
#df = df[df["text"].str.len() > 20]
#df = df[df["text"].str.split().str.len() >= 2]


In [8]:
# =========================
# 7. Remove duplicates
# =========================
df = df.drop_duplicates(subset=["text"])

In [9]:
# =========================
# 8. Convert rating
# =========================
df["rating"] = df["rating"].astype(int)

In [10]:
# =========================
# 9. Create sentiment label (optional but useful)
# =========================
def map_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["rating"].apply(map_sentiment)

In [11]:
# =========================
# 10. Final check
# =========================
print("Clean shape:", df.shape)
print(df["sentiment"].value_counts())

Clean shape: (10150, 4)
sentiment
positive    5174
negative    4035
neutral      941
Name: count, dtype: int64


In [12]:
# =========================
# 11. Save CLEAN dataset
# =========================
df.to_csv("../data/BMW/clean_reviews.csv", index=False)

print("✅ Clean dataset saved!")

✅ Clean dataset saved!
